In [ ]:

import sys
import torch
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

print("\nEnvironment is ready!")


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.14.1+cu130
GPU available: True
GPU name: Tesla T4

Environment is ready!


In [ ]:

import pandas as pd

# Download the Jigsaw dataset
url = "https://huggingface.co/datasets/Heliosoph/Jigsaw-Toxic-Comments/resolve/main/train.csv.gz"

df = pd.read_csv(url, compression="gzip")

# Keep only the columns we need
df = df[["comment_text", "toxic"]]

# Rename columns
df.columns = ["text", "label"]

# Remove missing values
df = df.dropna()

# Display dataset information
print("Dataset downloaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 comments:")
display(df.head())

print("\nClass distribution:")
print(df["label"].value_counts())


Dataset downloaded successfully!
Dataset shape: (159571, 2)

First 5 comments:


,text,label
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0



Class distribution:
label
0    144277
1     15294
Name: count, dtype: int64


In [ ]:

from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())


Training data shape: (127656,)
Testing data shape: (31915,)

Training class distribution:
label
0    115421
1     12235
Name: count, dtype: int64

Testing class distribution:
label
0    28856
1     3059
Name: count, dtype: int64


In [ ]:

import re

def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Replace repeated characters (soooo -> soo)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # Keep letters, numbers and basic punctuation
    text = re.sub(r"[^a-z0-9\s!?']", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply text cleaning to training and testing data
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

print("Text preprocessing completed!")

print("\nOriginal comment:")
print(X_train.iloc[0])

print("\nCleaned comment:")
print(X_train_clean.iloc[0])

print("\nTraining samples:", len(X_train_clean))
print("Testing samples:", len(X_test_clean))


Text preprocessing completed!

Original comment:
And more unfounded personal attacks by  here at Beeblebroxe's talk page. It just gets better and better. I suppose I am to blame for that conduct as well. Its somehow my fault I guess.

Cleaned comment:
and more unfounded personal attacks by here at beeblebroxe's talk page it just gets better and better i suppose i am to blame for that conduct as well its somehow my fault i guess

Training samples: 127656
Testing samples: 31915


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

# Transform training and testing text
X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf = tfidf.transform(X_test_clean)

print("TF-IDF transformation completed!")
print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)


In [ ]:

# Create Logistic Regression model
baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train the model
baseline_model.fit(X_train_tfidf, y_train)

print("Baseline model training completed!")


In [ ]:

# Make predictions on test data
y_pred_baseline = baseline_model.predict(X_test_tfidf)

# Calculate evaluation metrics
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline, average="macro")

print("BASELINE MODEL RESULTS")
print("----------------------")
print("Accuracy:", round(baseline_accuracy, 4))
print("Macro-F1:", round(baseline_f1, 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_baseline,
    target_names=["Clean", "Offensive"]
))


In [ ]:
!pip install -q gensim

In [ ]:
from gensim.models import Word2Vec

# Convert training comments into word lists
train_tokens = [text.split() for text in X_train_clean]

# Train our own Word2Vec model
word2vec_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=5
)

print("Word2Vec training completed!")
print("Vocabulary size:", len(word2vec_model.wv))
print("Vector size:", word2vec_model.vector_size)

In [ ]:
import numpy as np

def get_sentence_vector(sentence, model, vector_size=100):
    words = sentence.split()

    word_vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    if len(word_vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(word_vectors, axis=0)


X_train_w2v = np.array([
    get_sentence_vector(text, word2vec_model)
    for text in X_train_clean
])

X_test_w2v = np.array([
    get_sentence_vector(text, word2vec_model)
    for text in X_test_clean
])

print("Training data shape:", X_train_w2v.shape)
print("Testing data shape:", X_test_w2v.shape)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convert data into PyTorch tensors
X_train_tensor = torch.tensor(X_train_w2v, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_w2v, dtype=torch.float32)

y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.float32
).unsqueeze(1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32
).unsqueeze(1)

# Create training and testing datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset, batch_size=256, shuffle=True
)

test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False
)

print("PyTorch datasets created successfully!")
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

In [ ]:
class FeedforwardNN(nn.Module):
    def __init__(self):
        super(FeedforwardNN, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(100, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

nn_model = FeedforwardNN().to(device)

print(nn_model)
print("\nUsing device:", device)

In [ ]:
import torch.optim as optim

criterion = nn.BCELoss()

optimizer = optim.Adam(
    nn_model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    nn_model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = nn_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {average_loss:.4f}"
    )

print("\nNeural Network training completed!")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

nn_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)

        outputs = nn_model(inputs)
        predictions = (outputs >= 0.5).int()

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

y_pred_nn = np.array(all_predictions).flatten()
y_true_nn = np.array(all_labels).flatten()

nn_accuracy = accuracy_score(y_true_nn, y_pred_nn)
nn_f1 = f1_score(y_true_nn, y_pred_nn, average="macro")

print("NEURAL NETWORK RESULTS")
print("----------------------")
print("Accuracy:", round(nn_accuracy, 4))
print("Macro-F1:", round(nn_f1, 4))

print("\nClassification Report:")
print(classification_report(
    y_true_nn,
    y_pred_nn,
    target_names=["Clean", "Offensive"]
))

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained Transformer model
transformer_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

transformer_model = transformer_model.to(device)

print("Transformer model loaded successfully!")
print("Model:", model_name)
print("Device:", device)

In [ ]:
from transformers import DataCollatorWithPadding

# Tokenize training data
train_encodings = tokenizer(
    X_train_clean.tolist(),
    truncation=True,
    padding=False,
    max_length=128
)

# Tokenize testing data
test_encodings = tokenizer(
    X_test_clean.tolist(),
    truncation=True,
    padding=False,
    max_length=128
)

print("Tokenization completed!")
print("Training samples:", len(train_encodings["input_ids"]))
print("Testing samples:", len(test_encodings["input_ids"]))
print("Maximum sequence length: 128")

In [ ]:
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx], dtype=torch.long
        )

        return item


train_transformer_dataset = ToxicDataset(
    train_encodings, y_train
)

test_transformer_dataset = ToxicDataset(
    test_encodings, y_test
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Transformer datasets created successfully!")
print("Training samples:", len(train_transformer_dataset))
print("Testing samples:", len(test_transformer_dataset))

In [ ]:
from torch.utils.data import DataLoader, Subset

# Use a subset for faster Transformer training
train_subset = Subset(
    train_transformer_dataset,
    range(20000)
)

train_transformer_loader = DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

test_transformer_loader = DataLoader(
    test_transformer_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator
)

print("Transformer DataLoaders created!")
print("Training samples:", len(train_subset))
print("Testing samples:", len(test_transformer_dataset))
print("Training batches:", len(train_transformer_loader))
print("Testing batches:", len(test_transformer_loader))

In [ ]:
from torch.optim import AdamW

transformer_optimizer = AdamW(
    transformer_model.parameters(),
    lr=2e-5
)

num_transformer_epochs = 2

print("Transformer optimizer created!")
print("Learning rate: 2e-5")
print("Epochs:", num_transformer_epochs)

In [ ]:
num_transformer_epochs = 2

for epoch in range(num_transformer_epochs):
    transformer_model.train()
    total_loss = 0

    for batch in train_transformer_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        transformer_optimizer.zero_grad()

        outputs = transformer_model(**batch)
        loss = outputs.loss

        loss.backward()
        transformer_optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_transformer_loader)

    print(
        f"Epoch [{epoch+1}/{num_transformer_epochs}], "
        f"Loss: {average_loss:.4f}"
    )

print("\nTransformer training completed!")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

transformer_model.eval()

transformer_predictions = []
transformer_labels = []

with torch.no_grad():
    for batch in test_transformer_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = transformer_model(**batch)
        predictions = torch.argmax(outputs.logits, dim=1)

        transformer_predictions.extend(
            predictions.cpu().numpy()
        )

        transformer_labels.extend(
            batch["labels"].cpu().numpy()
        )

y_pred_transformer = np.array(transformer_predictions)
y_true_transformer = np.array(transformer_labels)

transformer_accuracy = accuracy_score(
    y_true_transformer, y_pred_transformer
)

transformer_f1 = f1_score(
    y_true_transformer,
    y_pred_transformer,
    average="macro"
)

print("TRANSFORMER RESULTS")
print("-------------------")
print("Accuracy:", round(transformer_accuracy, 4))
print("Macro-F1:", round(transformer_f1, 4))

print("\nClassification Report:")
print(classification_report(
    y_true_transformer,
    y_pred_transformer,
    target_names=["Clean", "Offensive"]
))

In [ ]:
import copy

# Keep the original Transformer unchanged
transformer_cpu = copy.deepcopy(
    transformer_model
).to("cpu")

transformer_cpu.eval()

print("Transformer copied to CPU successfully!")

In [ ]:
import torch

quantized_model = torch.quantization.quantize_dynamic(
    transformer_cpu,
    {torch.nn.Linear},
    dtype=torch.qint8
)

quantized_model.eval()

print("INT8 Quantization completed!")
print("Quantized model is ready on CPU.")

In [ ]:
import os
import torch

def get_model_size(model, filename):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / (1024 * 1024)
    os.remove(filename)
    return size_mb

original_size = get_model_size(
    transformer_cpu, "original_model.pth"
)

quantized_size = get_model_size(
    quantized_model, "quantized_model.pth"
)

print("ORIGINAL MODEL SIZE:", round(original_size, 2), "MB")
print("QUANTIZED MODEL SIZE:", round(quantized_size, 2), "MB")

reduction = (1 - quantized_size / original_size) * 100

print("Size Reduction:", round(reduction, 2), "%")

In [ ]:
from torch.utils.data import Subset, DataLoader

# Select 2,000 test samples
small_test_indices = list(range(2000))

small_test_dataset = Subset(
    test_transformer_dataset,
    small_test_indices
)

small_test_loader = DataLoader(
    small_test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator
)

print("Small test dataset created!")
print("Number of test samples:", len(small_test_dataset))
print("Number of batches:", len(small_test_loader))

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Optimize CPU execution
torch.set_num_threads(4)

# Use larger batches
fast_test_loader = DataLoader(
    test_transformer_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=data_collator
)

quantized_model.eval()

quantized_predictions = []
quantized_labels = []

print("Starting full test evaluation...")

with torch.no_grad():
    for i, batch in enumerate(fast_test_loader):

        labels = batch["labels"]

        inputs = {
            key: value
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = quantized_model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=1)

        quantized_predictions.extend(predictions.numpy())
        quantized_labels.extend(labels.numpy())

        if (i + 1) % 50 == 0:
            print(
                f"Processed {(i + 1) * 64} test samples..."
            )

y_pred_quantized = np.array(quantized_predictions)
y_true_quantized = np.array(quantized_labels)

quantized_accuracy = accuracy_score(
    y_true_quantized, y_pred_quantized
)

quantized_f1 = f1_score(
    y_true_quantized,
    y_pred_quantized,
    average="macro"
)

print("\nQUANTIZED TRANSFORMER RESULTS")
print("Accuracy:", round(quantized_accuracy, 4))
print("Macro-F1:", round(quantized_f1, 4))

print(classification_report(
    y_true_quantized,
    y_pred_quantized,
    target_names=["Clean", "Offensive"]
))

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Word2Vec + Feedforward Neural Network",
        "DistilBERT",
        "Quantized DistilBERT"
    ],
    "Accuracy": [0.9386, 0.9534, 0.9597, 0.9640],
    "Macro-F1": [0.8472, 0.8487, 0.8905, 0.8941],
    "Model Size (MB)": [
        None,
        None,
        255.45,
        132.29
    ]
})

display(results)

results.to_csv("model_comparison_results.csv", index=False)

print("Results saved successfully!")

In [ ]:
import time
import numpy as np

sample_text = X_test_clean.iloc[0]

latencies = []

for _ in range(100):
    start_time = time.perf_counter()

    baseline_model.predict(
        tfidf.transform([sample_text])
    )

    end_time = time.perf_counter()

    latencies.append((end_time - start_time) * 1000)

baseline_latency = np.mean(latencies)

print("TF-IDF Model Average Latency:", round(baseline_latency, 4), "ms")

In [ ]:
import time
import numpy as np

sample_vector = torch.tensor(
    X_test_w2v[0:1],
    dtype=torch.float32
).to(device)

nn_model.eval()

latencies = []

with torch.no_grad():
    for _ in range(100):
        start_time = time.perf_counter()

        nn_model(sample_vector)

        if device.type == "cuda":
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        latencies.append((end_time - start_time) * 1000)

nn_latency = np.mean(latencies)

print("Word2Vec + Neural Network Average Latency:",
      round(nn_latency, 4), "ms")

In [ ]:
import time
import numpy as np

transformer_model.eval()

sample_encoding = tokenizer(
    X_test_clean.iloc[0],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

sample_encoding = {
    key: value.to(device)
    for key, value in sample_encoding.items()
}

# Warm-up
with torch.no_grad():
    for _ in range(5):
        transformer_model(**sample_encoding)

if device.type == "cuda":
    torch.cuda.synchronize()

latencies = []

with torch.no_grad():
    for _ in range(100):
        start_time = time.perf_counter()

        transformer_model(**sample_encoding)

        if device.type == "cuda":
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        latencies.append((end_time - start_time) * 1000)

transformer_latency = np.mean(latencies)

print("DistilBERT Average Latency:",
      round(transformer_latency, 4), "ms")

In [ ]:
import time
import numpy as np

quantized_model.eval()

quantized_sample = {
    key: value.cpu()
    for key, value in sample_encoding.items()
}

# Warm-up
with torch.no_grad():
    for _ in range(5):
        quantized_model(**quantized_sample)

latencies = []

with torch.no_grad():
    for _ in range(100):
        start_time = time.perf_counter()

        quantized_model(**quantized_sample)

        end_time = time.perf_counter()

        latencies.append((end_time - start_time) * 1000)

quantized_latency = np.mean(latencies)

print("Quantized Transformer Average Latency:",
      round(quantized_latency, 4), "ms")

In [ ]:
results["Average Latency (ms)"] = [
    baseline_latency,
    nn_latency,
    transformer_latency,
    quantized_latency
]

display(results)

results.to_csv("final_model_comparison.csv", index=False)

print("Final comparison table saved successfully!")

In [ ]:
import os
import joblib
import torch

# Classical model: TF-IDF + Logistic Regression
joblib.dump(
    (tfidf, baseline_model),
    "classical_model.pkl"
)

classical_size = os.path.getsize(
    "classical_model.pkl"
) / (1024 * 1024)

# Neural model: Word2Vec + Feedforward Neural Network
word2vec_model.save("word2vec.model")
torch.save(nn_model.state_dict(), "neural_network.pth")

word2vec_size = os.path.getsize(
    "word2vec.model"
) / (1024 * 1024)

neural_network_size = os.path.getsize(
    "neural_network.pth"
) / (1024 * 1024)

neural_size = word2vec_size + neural_network_size

# Update comparison table
results.loc[0, "Model Size (MB)"] = classical_size
results.loc[1, "Model Size (MB)"] = neural_size

print("Classical Model Size:", round(classical_size, 2), "MB")
print("Neural Network Model Size:", round(neural_size, 2), "MB")

display(results)

In [ ]:
!pip install -q gradio

In [ ]:
import torch
import gradio as gr
import numpy as np

quantized_model.eval()

def predict_moderation(text):

    if not text or not text.strip():
        return {"Please enter a message": 1.0}

    cleaned_text = clean_text(text)

    inputs = tokenizer(
        cleaned_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = quantized_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)[0]

    labels = {
        0: "Clean",
        1: "Offensive"
    }

    results = {
        labels[i]: float(probabilities[i])
        for i in range(2)
    }

    return results

print("Moderation prediction function created successfully!")

In [ ]:
demo = gr.Interface(
    fn=predict_moderation,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type your message here...",
        label="Enter Your Message"
    ),
    outputs=gr.Label(
        num_top_classes=2,
        label="Moderation Results"
    ),
    title="Real-Time Chat Moderation",
    description=(
        "An intelligent application that detects "
        "offensive messages using a Quantized Transformer model."
    ),
    examples=[
        ["Have a wonderful day!"],
        ["You are a terrible person."],
        ["Thank you for your help."]
    ]
)

print("Gradio interface created successfully!")

In [69]:
print(demo)
print(demo.server_port)

Gradio Interface for: predict_moderation
----------------------------------------
inputs:
|-<gradio.components.textbox.Textbox object at 0x78c64affc590>
outputs:
|-<gradio.components.label.Label object at 0x78c64afa5fd0>
7860


In [70]:
import requests

response = requests.get("http://127.0.0.1:7860")
print(response.status_code)

200


In [ ]:
requirements = """
pandas
numpy
scikit-learn
gensim
torch
transformers
gradio
"""

with open("requirements.txt", "w") as file:
    file.write(requirements.strip())

print("requirements.txt created successfully!")

with open("requirements.txt", "r") as file:
    print(file.read())

In [ ]:
readme_content = """# Real-Time Chat Moderation

## Project Description
An intelligent application for detecting offensive comments using machine learning and transformer models.

## Models
1. TF-IDF + Logistic Regression
2. Word2Vec + Feedforward Neural Network
3. DistilBERT
4. Quantized DistilBERT

## Model Performance

| Model | Accuracy | Macro-F1 |
|---|---:|---:|
| TF-IDF + Logistic Regression | 0.9386 | 0.8472 |
| Word2Vec + Neural Network | 0.9534 | 0.8487 |
| DistilBERT | 0.9597 | 0.8905 |
| Quantized DistilBERT | 0.9640 | 0.8941 |

## Quantization
Model size was reduced from 255.45 MB to 132.29 MB, achieving a 48.21% reduction.

## Application
The application uses Gradio and Quantized DistilBERT for real-time chat moderation.

## Technologies
Python, PyTorch, Scikit-learn, Gensim, Hugging Face Transformers, Gradio.

## Installation
Run: pip install -r requirements.txt

## Dataset
Jigsaw Toxic Comment Classification Dataset.

## Project
SWE015
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("README.md created successfully!")

In [ ]:
import os
import shutil

project_folder = "SWE015_Chat_Moderation"

os.makedirs(project_folder, exist_ok=True)

# Copy project files
shutil.copy("README.md", project_folder)
shutil.copy("requirements.txt", project_folder)

# Copy the notebook
notebook_files = [
    f for f in os.listdir("/content")
    if f.endswith(".ipynb")
]

print("Available notebooks:", notebook_files)

In [ ]:
import torch
import os

quantized_model.eval()

torch.save(
    quantized_model,
    "quantized_model.pt"
)

tokenizer.save_pretrained("moderation_tokenizer")

print("Quantized model saved successfully!")
print("Model size:",
      round(os.path.getsize("quantized_model.pt") / (1024 * 1024), 2),
      "MB")

In [ ]:
import os
import shutil

project_folder = "/content/SWE015_Chat_Moderation"

# Copy quantized model
shutil.copy2(
    "quantized_model.pt",
    project_folder
)

# Copy tokenizer folder
shutil.copytree(
    "moderation_tokenizer",
    os.path.join(project_folder, "moderation_tokenizer"),
    dirs_exist_ok=True
)

print("Project files:")
for item in os.listdir(project_folder):
    print("-", item)

print("\nModel and tokenizer copied successfully!")

In [ ]:
app_code = '''
import torch
import gradio as gr
import re
from transformers import AutoTokenizer

MODEL_PATH = "quantized_model.pt"
TOKENIZER_PATH = "moderation_tokenizer"

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\\\\S+|www\\\\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"(.)\\\\1{2,}", r"\\\\1\\\\1", text)
    text = re.sub(r"[^a-z0-9\\\\s!?']", " ", text)
    text = re.sub(r"\\\\s+", " ", text).strip()
    return text

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
model = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
model.eval()

def predict_moderation(text):
    if not text or not text.strip():
        return {"Please enter a message": 1.0}

    cleaned_text = clean_text(text)
    inputs = tokenizer(
        cleaned_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)[0]

    return {
        "Clean": float(probabilities[0]),
        "Offensive": float(probabilities[1])
    }

demo = gr.Interface(
    fn=predict_moderation,
    inputs=gr.Textbox(lines=4, label="Enter Your Message"),
    outputs=gr.Label(num_top_classes=2, label="Moderation Results"),
    title="Real-Time Chat Moderation",
    description="Detect offensive messages using a Quantized Transformer model."
)

demo.launch()
'''

app_path = "/content/SWE015_Chat_Moderation/app.py"

with open(app_path, "w", encoding="utf-8") as file:
    file.write(app_code)

print("app.py created successfully!")

In [ ]:
import os

project_folder = "/content/SWE015_Chat_Moderation"

for root, dirs, files in os.walk(project_folder):
    level = root.replace(project_folder, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

In [ ]:
import shutil

project_folder = "/content/SWE015_Chat_Moderation"

zip_path = shutil.make_archive(
    "/content/SWE015_Chat_Moderation",
    "zip",
    project_folder
)

print("ZIP created successfully!")
print(zip_path)

In [ ]:
from google.colab import files

files.download("/content/SWE015_Chat_Moderation.zip")

In [76]:
demo.launch(share=True, debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9b84fdafd703fe3881.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

In [77]:
demo.close()

Closing server running on port: 7860


In [74]:
import gradio
print("Gradio version:", gradio.__version__)
print("Server running:", demo.server_port)

Gradio version: 6.31.0
Server running: 7860


In [75]:
import requests

url = "https://9b84fdafd703fe3881.gradio.live"
response = requests.get(url)
print(response.status_code)
print(response.text[:300])

404

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>No interface is running</title>
</head>
<style>
    body{
    font-family: 'Lato', sans-serif;
    color: #888;
    text-align: center;
    }
    #main{
        padding-top:30px
    }
</style>
<body>

<div id="main">
    


In [80]:
demo.launch(
    share=True,
    debug=False,
    prevent_thread_lock=True
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b84fdafd703fe3881.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

In [79]:
import requests

response = requests.get("https://9b84fdafd703fe3881.gradio.live")
print(response.status_code)

404


In [81]:
demo.close()

Closing server running on port: 7860


In [82]:
!pip install -q gradio==5.49.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.13.3 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.7.1 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.


In [83]:
!pip show gradio

Name: gradio
Version: 5.49.1
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License-Expression: Apache-2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiofiles, anyio, audioop-lts, brotli, fastapi, ffmpy, gradio-client, groovy, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pyyaml, ruff, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 


In [84]:
import gradio
print(gradio.__version__)

6.31.0


In [1]:
import gradio
print(gradio.__version__)

5.49.1


In [2]:
import torch
import gradio as gr
import numpy as np
from transformers import AutoTokenizer

print("Gradio:", gr.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.is_available())

Gradio: 5.49.1
PyTorch: 2.14.1+cu130
GPU: True


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "quantized_model.pt":
            print(os.path.join(root, file))

In [7]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "onurtaysun/SWE015-Chat-Moderation-2026"

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename="quantized_model.pt"
)

print("Model downloaded:", model_path)

quantized_model.pt: reconstructing file:   0%|          |  0.00B /  139MB            

quantized_model.pt: downloading bytes:           |  0.00B            

Model downloaded: /root/.cache/huggingface/hub/models--onurtaysun--SWE015-Chat-Moderation-2026/snapshots/ceb38e5710d4ac185728b6b203218d6a1af7b878/quantized_model.pt


In [8]:
quantized_model = torch.load(
    model_path,
    map_location="cpu",
    weights_only=False
)

quantized_model.eval()

print("Quantized model loaded successfully!")


/usr/local/lib/python3.13/dist-packages/torch/_utils.py:466: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,
/usr/local/lib/python3.13/dist-packages/torch/_utils.py:461: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  tensor = torch._empty_affine_quantized(


Quantized model loaded successfully!


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "onurtaysun/SWE015-Chat-Moderation-2026"
)

print("Tokenizer loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [10]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^a-z0-9\s!?']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def predict_moderation(text):
    if not text or not text.strip():
        return {"Please enter a message": 1.0}

    cleaned_text = clean_text(text)

    inputs = tokenizer(
        cleaned_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = quantized_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)[0]

    return {
        "Clean": float(probabilities[0]),
        "Offensive": float(probabilities[1])
    }

print("Prediction function ready!")

Prediction function ready!


In [ ]:
demo = gr.Interface(
    fn=predict_moderation,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type your message here...",
        label="Enter Your Message"
    ),
    outputs=gr.Label(
        num_top_classes=2,
        label="Moderation Results"
    ),
    title="Real-Time Chat Moderation",
    description="An intelligent application that detects offensive messages using a Quantized Transformer model.",
    examples=[
        ["Have a wonderful day!"],
        ["You are a terrible person."],
        ["Thank you for your help."]
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://616b27179c46c920b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
